# MNIST Digit Classification using CNN

This notebook builds a clean Convolutional Neural Network (CNN) to classify handwritten digits from the MNIST dataset.

The workflow is:

1. Import libraries  
2. Load MNIST dataset  
3. Explore the data  
4. Preprocess images and labels  
5. Create train-validation split  
6. Build CNN model  
7. Train the model  
8. Evaluate performance  
9. Generate predictions  
10. Test inference on a single image  


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

## 2. Load Dataset

In [ ]:
# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("Training images shape:", x_train.shape)
print("Training labels shape:", y_train.shape)
print("Testing images shape:", x_test.shape)
print("Testing labels shape:", y_test.shape)

MNIST contains grayscale handwritten digit images.

Each image has the shape:

```text
28 x 28 pixels
```

Each label is one digit from:

```text
0 to 9
```

## 3. Check Class Distribution

In [ ]:
unique_train, counts_train = np.unique(y_train, return_counts=True)
unique_test, counts_test = np.unique(y_test, return_counts=True)

print("Train label distribution:")
print(dict(zip(unique_train, counts_train)))

print("\nTest label distribution:")
print(dict(zip(unique_test, counts_test)))

Checking class distribution helps us verify whether the dataset has enough examples for each digit.

A balanced dataset helps the model learn all classes fairly.

## 4. Visualize Sample Images

In [ ]:
plt.figure(figsize=(10, 6))

random_indices = np.random.randint(0, x_train.shape[0], size=30)

for i, idx in enumerate(random_indices):
    plt.subplot(5, 6, i + 1)
    plt.imshow(x_train[idx], cmap="gray")
    plt.title(f"Label: {y_train[idx]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 5. Preprocess Images

CNN layers expect image data in this format:

```text
(number_of_images, height, width, channels)
```

For MNIST:

```text
height = 28
width = 28
channels = 1
```

The channel value is `1` because MNIST images are grayscale.

We also normalize pixel values from `0–255` to `0–1`, which helps the neural network train better.

In [ ]:
# Reshape images to include channel dimension
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32")
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32")

# Normalize pixel values
x_train = x_train / 255.0
x_test = x_test / 255.0

print("New training shape:", x_train.shape)
print("New testing shape:", x_test.shape)
print("Pixel range:", x_train.min(), "to", x_train.max())

## 6. One-Hot Encode Labels

The model will output 10 probabilities, one for each digit.

So labels must be converted from single numbers into one-hot encoded vectors.

Example:

```text
Original label: 7

One-hot label:
[0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
```

In [ ]:
y_cat_train = to_categorical(y_train, num_classes=10)
y_cat_test = to_categorical(y_test, num_classes=10)

print("Original label example:", y_train[0])
print("One-hot encoded example:", y_cat_train[0])

## 7. Train-Validation Split

We split the training data into:

- Training data: used to train the model
- Validation data: used to monitor performance during training
- Test data: used only at the end for final evaluation

The test set should not be used as validation data during training.

In [ ]:
x_train_final, x_valid, y_train_final, y_valid = train_test_split(
    x_train,
    y_cat_train,
    test_size=0.2,
    random_state=22,
    stratify=y_train
)

print("Final training data:", x_train_final.shape)
print("Validation data:", x_valid.shape)

## 8. Build CNN Model

The CNN architecture is:

```text
Input image
   ↓
Conv2D
   ↓
MaxPooling
   ↓
Flatten
   ↓
Dense hidden layer
   ↓
Dropout
   ↓
Softmax output layer
```

The final layer has 10 neurons because there are 10 digit classes.

In [ ]:
model = Sequential()

model.add(
    Conv2D(
        filters=32,
        kernel_size=(4, 4),
        activation="relu",
        input_shape=(28, 28, 1)
    )
)

model.add(MaxPool2D(pool_size=(2, 2)))

model.add(Flatten())

model.add(Dense(128, activation="relu"))

model.add(Dropout(0.3))

model.add(Dense(10, activation="softmax"))

## 9. Model Summary

In [ ]:
model.summary()

## 10. Compile Model

We use:

- `categorical_crossentropy` because this is a multiclass classification problem
- `adam` optimizer to update model weights
- `accuracy` to track classification performance

In [ ]:
model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

## 11. Early Stopping

Early stopping stops training when validation loss stops improving.

`restore_best_weights=True` makes sure the final model keeps the best validation performance.

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

## 12. Train Model

In [ ]:
history = model.fit(
    x_train_final,
    y_train_final,
    epochs=10,
    batch_size=128,
    validation_data=(x_valid, y_valid),
    callbacks=[early_stop]
)

## 13. Training History

In [ ]:
history_df = pd.DataFrame(history.history)
history_df

## 14. Plot Loss

In [ ]:
history_df[["loss", "val_loss"]].plot(figsize=(8, 5))
plt.title("Training Loss vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

## 15. Plot Accuracy

In [ ]:
history_df[["accuracy", "val_accuracy"]].plot(figsize=(8, 5))
plt.title("Training Accuracy vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True)
plt.show()

## 16. Evaluate on Test Data

Now we evaluate the model on the test set.

This gives the final performance on unseen data.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_cat_test)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

## 17. Generate Predictions

In [ ]:
y_pred_probs = model.predict(x_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

print("Prediction probabilities shape:", y_pred_probs.shape)
print("Predicted classes shape:", y_pred_classes.shape)

## 18. Classification Report

In [ ]:
print(classification_report(y_test, y_pred_classes))

## 19. Confusion Matrix

The confusion matrix shows where the model is making correct and incorrect predictions.

The diagonal values represent correct predictions.

In [ ]:
cm = confusion_matrix(y_test, y_pred_classes)

plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.colorbar()

for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.show()

## 20. Single Image Inference

In [ ]:
image_index = 225

sample_image = x_test[image_index]
true_label = y_test[image_index]

plt.imshow(sample_image.reshape(28, 28), cmap="gray")
plt.title(f"True Label: {true_label}")
plt.axis("off")
plt.show()

sample_image_input = sample_image.reshape(1, 28, 28, 1)

prediction_probs = model.predict(sample_image_input)
predicted_label = np.argmax(prediction_probs)

print("True Label:", true_label)
print("Predicted Label:", predicted_label)
print("Prediction Probabilities:", prediction_probs)

## Final Notes

This notebook trains a CNN to classify handwritten digits from the MNIST dataset.

Key cleanup improvements included:

- Removed unused imports
- Normalized pixel values
- Used a proper train-validation-test split
- Avoided using test data during training
- Added dropout to reduce overfitting
- Added `restore_best_weights=True` in early stopping
- Completed single-image inference properly
